In [14]:
import pandas as pd

all_files = [
    'Bike Share Toronto Ridership_Q1 2018.csv',
    'Bike Share Toronto Ridership_Q2 2018.csv',
    'Bike Share Toronto Ridership_Q3 2018.csv',
    'Bike Share Toronto Ridership_Q4 2018.csv',
]

In [15]:
df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
print(f"Total rows: {df.shape[0]:,}")

Total rows: 1,922,955


In [17]:
df.head()

,trip_id,trip_duration_seconds,from_station_id,trip_start_time,from_station_name,trip_stop_time,to_station_id,to_station_name,user_type
0,2383648,393,7018,1/1/2018 0:47,Bremner Blvd / Rees St,1/1/2018 0:54,7176,Bathurst St / Fort York Blvd,Annual Member
1,2383649,625,7184,1/1/2018 0:52,Ossington Ave / College St,1/1/2018 1:03,7191,Central Tech (Harbord St),Annual Member
2,2383650,233,7235,1/1/2018 0:55,Bay St / College St (West Side) - SMART,1/1/2018 0:59,7021,Bay St / Albert St,Annual Member
3,2383651,1138,7202,1/1/2018 0:57,Queen St W / York St (City Hall),1/1/2018 1:16,7020,Phoebe St / Spadina Ave,Annual Member
4,2383652,703,7004,1/1/2018 1:00,University Ave / Elm St,1/1/2018 1:12,7060,Princess St / Adelaide St E,Annual Member


In [16]:
# Overview
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df['user_type'].value_counts())
print(df['trip_duration_seconds'].describe())

(1922955, 9)
trip_id                   int64
trip_duration_seconds     int64
from_station_id           int64
trip_start_time          object
from_station_name        object
trip_stop_time           object
to_station_id             int64
to_station_name          object
user_type                object
dtype: object
trip_id                  0
trip_duration_seconds    0
from_station_id          0
trip_start_time          0
from_station_name        0
trip_stop_time           0
to_station_id            0
to_station_name          0
user_type                0
dtype: int64
user_type
Annual Member    1572980
Casual Member     349975
Name: count, dtype: int64
count    1.922955e+06
mean     9.629760e+02
std      1.595530e+03
min      6.000000e+01
25%      4.220000e+02
50%      6.700000e+02
75%      1.051000e+03
max      5.507700e+04
Name: trip_duration_seconds, dtype: float64


In [18]:
# Fix datetime
df['trip_start_time'] = pd.to_datetime(df['trip_start_time'])
df['trip_stop_time'] = pd.to_datetime(df['trip_stop_time'])

# Standardize user_type
df['user_type'] = df['user_type'].replace({
    'Annual Member': 'Member',
    'Casual Member': 'Casual'
})

# Extract time features
df['year'] = df['trip_start_time'].dt.year
df['month'] = df['trip_start_time'].dt.month
df['day_of_week'] = df['trip_start_time'].dt.day_name()
df['hour'] = df['trip_start_time'].dt.hour

# Filter bad durations (keep 1 min to 24 hours)
df = df[(df['trip_duration_seconds'] >= 60) & (df['trip_duration_seconds'] <= 86400)]

# Confirm
print(df.shape)
print(df['user_type'].value_counts())
print(df['trip_start_time'].dtype)

(1922955, 13)
user_type
Member    1572980
Casual     349975
Name: count, dtype: int64
datetime64[ns]


In [19]:
df.head()

,trip_id,trip_duration_seconds,from_station_id,trip_start_time,from_station_name,trip_stop_time,to_station_id,to_station_name,user_type,year,month,day_of_week,hour
0,2383648,393,7018,2018-01-01 00:47:00,Bremner Blvd / Rees St,2018-01-01 00:54:00,7176,Bathurst St / Fort York Blvd,Member,2018,1,Monday,0
1,2383649,625,7184,2018-01-01 00:52:00,Ossington Ave / College St,2018-01-01 01:03:00,7191,Central Tech (Harbord St),Member,2018,1,Monday,0
2,2383650,233,7235,2018-01-01 00:55:00,Bay St / College St (West Side) - SMART,2018-01-01 00:59:00,7021,Bay St / Albert St,Member,2018,1,Monday,0
3,2383651,1138,7202,2018-01-01 00:57:00,Queen St W / York St (City Hall),2018-01-01 01:16:00,7020,Phoebe St / Spadina Ave,Member,2018,1,Monday,0
4,2383652,703,7004,2018-01-01 01:00:00,University Ave / Elm St,2018-01-01 01:12:00,7060,Princess St / Adelaide St E,Member,2018,1,Monday,1


In [20]:
df.to_csv('toronto_bikes_clean.csv', index=False)
